# Parsing Complex PDFs/Documents Locally

Standard PDF readers often mangle tables, multi-column layouts, and embedded images into a jumbled stream of text. `pymupdf4llm` is a local, offline PDF-to-markdown converter (built on PyMuPDF) that preserves table structure — no cloud service or API key required.


**Step 1 — Set up the usual LLM/embedding models.** No parsing-specific API key is needed — `pymupdf4llm` runs entirely locally.


In [1]:
# pip install pymupdf4llm  (already in requirements.txt)
import logging

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Quiet down noisy INFO-level logs from the HTTP client and LlamaIndex itself.
for noisy_logger in ("httpx", "llama_index"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Loads .env into os.environ so OPENAI_API_KEY is available.
load_dotenv()

Settings.llm = OpenAI(model="gpt-4.1-nano")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

**Step 2 — Parse the PDF with `pymupdf4llm`.** `pymupdf4llm.to_markdown()` reads `data/sample.pdf` locally and returns a clean markdown rendering, including tables — no upload, no network call, no API key.


In [2]:
import pymupdf4llm

# Parses sample.pdf entirely locally and returns the whole document as markdown
# text, with table structure preserved.
parsed_markdown: str = pymupdf4llm.to_markdown("data/sample.pdf")

print("--- pymupdf4llm markdown output (first 1200 chars) ---")
print(parsed_markdown[:1200])


=== Document parser messages ===
Using Tesseract for OCR processing.
--- pymupdf4llm markdown output (first 1200 chars) ---
# **Anime Protagonist Power Comparison Sheet** 

This reference sheet compares the protagonists of five landmark shonen and seinen anime series, spanning classic 2000s battle anime through the recent webtoon-adaptation boom. Each protagonist reaches their strongest known form through a distinct path — inherited power, martial training, a system-driven leveling mechanic, supernatural rules, or an evolving breathing technique — making direct power comparisons a common source of fan debate. 

The table below lists each character's signature technique, the studio behind their anime adaptation, and the year their series first aired, as a quick-reference sheet rather than a definitive power ranking. 

## **Protagonist Comparison Table** 

|**Character**|**Anime**|**Signature Technique**|**Studio**|**First Aired**|
|---|---|---|---|---|
|Naruto Uzumaki|Naruto|Rasengan|P

**Step 3 — Compare against a basic PDF reader.** `SimpleDirectoryReader` extracts the same PDF's text with no table awareness at all, so we can see exactly what `pymupdf4llm` improves on.


In [3]:
from llama_index.core import SimpleDirectoryReader

# The plain SimpleDirectoryReader extracts text but has no idea the PDF contains
# a table — it just flattens everything into one text stream.
basic_documents = SimpleDirectoryReader(input_files=["data/sample.pdf"]).load_data()
basic_text = "\n".join(document.text for document in basic_documents)

print("--- Basic PDF reader output (first 1200 chars) ---")
print(basic_text[:1200])

print("\nNotice the comparison table above stays readable as markdown rows in the ")
print("pymupdf4llm output, while the basic reader flattens it into a run of ")
print("loose names and words with no table structure.")

--- Basic PDF reader output (first 1200 chars) ---
Anime Protagonist Power Comparison Sheet
This reference sheet compares the protagonists of five landmark shonen and seinen anime series,
spanning classic 2000s battle anime through the recent webtoon-adaptation boom. Each protagonist
reaches their strongest known form through a distinct path — inherited power, martial training, a
system-driven leveling mechanic, supernatural rules, or an evolving breathing technique — making
direct power comparisons a common source of fan debate.
The table below lists each character's signature technique, the studio behind their anime adaptation,
and the year their series first aired, as a quick-reference sheet rather than a definitive power ranking.
Protagonist Comparison Table
Character
Anime
Signature Technique
Studio
First Aired
Naruto Uzumaki
Naruto
Rasengan
Pierrot
2002
Son Goku
Dragon Ball
Kamehameha
Toei Animation
1986
Sung Jin-Woo
Solo Leveling
Shadow Extraction
A-1 Pictures
2024
Light Yagami


**Step 4 — Index and query the parsed text.** The `pymupdf4llm` markdown gets wrapped in a `Document` and indexed like any other text — the parser choice only affects input quality, not the rest of the pipeline.


In [4]:
from llama_index.core import Document, VectorStoreIndex

document = Document(text=parsed_markdown)
index = VectorStoreIndex.from_documents([document])

response = index.as_query_engine().query(
    "Which studio animated Tanjiro Kamado's anime series, and what year did it first air?"
)
print(response)

ufotable animated Tanjiro Kamado's anime series, and it first aired in 2019.


### Summary

- A table-aware local parser like `pymupdf4llm` matters most for table-heavy, multi-column, or otherwise complex documents — for simple plain-text files, a basic reader is fine and free.
- The rest of the pipeline (building an index, querying it) doesn't change at all based on which parser you used — `pymupdf4llm` just gives the index cleaner input to work with, entirely offline.
